# Crypto Perpetuals Funding: Holdout Backtest

**Chapter 20 - Out-of-sample evaluation**

[`17_holdout_predictions`](17_holdout_predictions.ipynb) refitted the selected
configuration on history ending before 2024 and wrote its predictions over 2024-25. This
notebook trades them with the sizing and the overlay that configuration carries, and
registers the result under `stage='holdout'`.

**Prerequisites:** [`17_holdout_predictions`](17_holdout_predictions.ipynb). This notebook
does not fit; if the prediction set it needs is absent, it raises rather than producing
one.

**Scope:** one backtest. The comparison against validation, and the correction that
comparison needs, are [`19_strategy_analysis`](19_strategy_analysis.ipynb)'s.

<!-- ML4T_ZH_BEGIN c0000-p000 -->
# 加密貨幣永續合約資金費：留出樣本回測

**第 20 章——樣本外評估**

[`17_holdout_predictions`](17_holdout_predictions.ipynb) 以 2024 年以前結束的歷史資料重新擬合選定配置，並寫出其在 2024 至 2025 年的預測。本 Notebook 使用該配置所帶的部位規模設定及風險覆蓋交易這些預測，並將結果登錄於 `stage='holdout'`。

**先備條件：** [`17_holdout_predictions`](17_holdout_predictions.ipynb)。本 Notebook 不進行擬合；若缺少所需的預測組，會拋出錯誤，而非自行產生一組。

**範圍：** 一次回測。與驗證結果的比較，以及該比較所需的修正，屬於 [`19_strategy_analysis`](19_strategy_analysis.ipynb)。
<!-- ML4T_ZH_END c0000-p000 -->


In [1]:
"""Crypto Perpetuals Funding: Holdout Backtest."""

import json
import sqlite3
import warnings

import polars as pl

warnings.filterwarnings("ignore")

from case_studies.crypto_perps_funding.funding_data import funding_rates_for_prices
from case_studies.research import open_study
from case_studies.research.holdout import build_holdout_training_spec
from case_studies.research.strategy import strategy_warmup_periods
from case_studies.utils.artifact_digest import value_digest
from case_studies.utils.backtest_loaders import get_backtest_config, load_backtest_prices_for
from case_studies.utils.backtest_presets import (
    ensure_backtest_spec,
    strategy_view,
)
from case_studies.utils.backtest_runner import resolved_allow_short_selling, run_backtest
from case_studies.utils.conformal import (
    compute_holdout_conformal_widths,
    ensure_conformal_calibration_identity,
    holdout_conformal_embargo_steps,
)
from case_studies.utils.registry import (
    backtest_run_status,
    read_predictions,
    training_hash_from_spec,
)
from case_studies.utils.strategy_analysis import resolve_solvent_carrier
from utils.paths import get_case_study_dir

In [2]:
CASE_STUDY_ID = "crypto_perps_funding"
EXECUTION_TIER = "canonical"
WORKSPACE: str = ""
MAX_SYMBOLS = 0
# Whether a holdout backtest of a DIFFERENT strategy may be superseded by this run. Off by
# default, and the same switch `17_holdout_predictions` uses for the model side.
REPLACE_HOLDOUT = False

In [3]:
study = open_study(CASE_STUDY_ID, execution_tier=EXECUTION_TIER, workspace=WORKSPACE or None)
CASE_DIR = get_case_study_dir(CASE_STUDY_ID)
bt_config = get_backtest_config(CASE_STUDY_ID)


def _registered_holdout_backtests(case_dir, prediction_hash):
    """The backtest hashes already registered against one holdout prediction set."""
    with sqlite3.connect(str(case_dir / "run_log" / "registry.db")) as conn:
        rows = conn.execute(
            "SELECT backtest_hash FROM backtest_runs WHERE prediction_hash = ? "
            "ORDER BY backtest_hash",
            (prediction_hash,),
        ).fetchall()
    return [{"backtest_hash": backtest_hash} for (backtest_hash,) in rows]


def _delete_holdout_backtest(case_dir, backtest_hash):
    """Remove one registered holdout backtest and the rows derived from it.

    Same rule as `17_holdout_predictions`' replacement of a superseded generation: a holdout
    result that has been observed and then left readable beside its replacement is still a
    number someone can quote.
    """
    with sqlite3.connect(str(case_dir / "run_log" / "registry.db")) as conn:
        conn.execute(
            "DELETE FROM backtest_paired_metrics WHERE challenger_hash = ? OR benchmark_hash = ?",
            (backtest_hash, backtest_hash),
        )
        conn.execute("DELETE FROM backtest_metrics WHERE backtest_hash = ?", (backtest_hash,))
        conn.execute("DELETE FROM backtest_runs WHERE backtest_hash = ?", (backtest_hash,))

## 1. The configuration, and the predictions it produced on the holdout

The selected configuration is resolved the same way [`16_costs`](16_costs.ipynb) and
[`17_holdout_predictions`](17_holdout_predictions.ipynb) resolve it, so all three run the
same configuration by construction rather than by a hash copied between them.

Which holdout prediction set belongs to it is derived rather than searched for. Re-deriving the
holdout training specification reproduces the training identity 17 registered - the derivation is
deterministic and the identity covers it - so the prediction set is looked up by that identity
and the selected configuration's checkpoint. A search over holdout prediction sets would have to
guess which one belonged to this configuration.

<!-- ML4T_ZH_BEGIN c0004-p000 -->
## 1. 配置及其在留出樣本上產生的預測

解析選定配置的方式與 [`16_costs`](16_costs.ipynb) 及 [`17_holdout_predictions`](17_holdout_predictions.ipynb) 相同，因此三者依相同規則執行同一項配置，而非靠彼此複製的雜湊值來維持一致。

屬於該配置的留出樣本預測組是透過推導取得，而非搜尋取得。重新推導留出樣本訓練規格，會重現第 17 份 Notebook 登錄的訓練識別資訊，因為推導具有決定性且識別資訊涵蓋該推導。因此，會依此識別資訊及選定配置的檢查點查找預測組。若搜尋所有留出樣本預測組，就得猜測哪一組屬於這項配置。
<!-- ML4T_ZH_END c0004-p000 -->


In [4]:
carrier = resolve_solvent_carrier(CASE_STUDY_ID)
LABEL = carrier["label"]
validation_prediction_record = study.results.open(carrier["val_prediction_hash"]).registry_record()

holdout_spec = build_holdout_training_spec(
    study,
    study.results.open(carrier["training_hash"]).spec(),
    timeline=(
        pl.read_parquet(study.root / "labels" / f"{LABEL}.parquet")
        .get_column("timestamp")
        .unique()
        .sort()
        .to_list()
    ),
    case_study=CASE_STUDY_ID,
)
holdout_training_hash = training_hash_from_spec(holdout_spec)

with sqlite3.connect(str(CASE_DIR / "run_log" / "registry.db")) as conn:
    match = conn.execute(
        """
        SELECT prediction_hash FROM prediction_sets
        WHERE split = 'holdout' AND training_hash = ?
          AND checkpoint_kind IS ? AND checkpoint_value IS ?
        """,
        (
            holdout_training_hash,
            validation_prediction_record["checkpoint_kind"],
            validation_prediction_record["checkpoint_value"],
        ),
    ).fetchone()
if match is None:
    raise RuntimeError(
        f"No holdout prediction set for training {holdout_training_hash}. Run "
        "17_holdout_predictions first; this notebook does not fit."
    )
HOLDOUT_PREDICTION_HASH = match[0]

print(f"Selected configuration: {carrier['val_backtest_hash']}  {carrier['config_name']} ({LABEL})")
print(f"Holdout training:   {holdout_training_hash}")
print(f"Holdout prediction: {HOLDOUT_PREDICTION_HASH}")

Selected configuration: 9f1fe71d18e6  enet_f0.03 (fwd_ret_24h)
Holdout training:   918aeaf5a811
Holdout prediction: 9adfb97d8db7


## 2. What the allocator needs before the window opens

This configuration allocates by `mvo_ledoit_wolf`, which is a moment estimator: it reads a
rolling window of underlying prices and produces weights from their covariance. That window
does not restart because the evaluation period does. Loading the holdout slice alone would
leave the first rebalance with no history to estimate from, and the loader would fall back
to a median-imputed warmup - so the first weeks of the holdout would be traded on weights
that describe nothing, and the resulting Sharpe would be a measurement of the fallback.

So the prices are loaded with the same lookback the allocator was given on validation.
`setup.yaml` declares `allocator_lookback: 240`, which is 240 eight-hourly bars, about
eighty days. The loader leaves the start of the window unconstrained by that much and still
caps the end at the canonical window end; the extra prefix is consumed by the rolling
window and does not enter return aggregation, because the engine only aggregates over the
rebalance timestamps the predictions carry.

The conformal branch below is inert for this configuration and is kept because the selected
configuration is resolved rather than fixed. `conformal_weighted` sizes from residuals the model
has already made, and on the holdout there are none to use - every holdout return realises inside
the window being evaluated - so a conformal configuration would calibrate from validation
residuals with an embargo covering the label horizon. This one allocates from price moments and
needs no calibration at all.

<!-- ML4T_ZH_BEGIN c0006-p000 -->
## 2. 時間窗開始前資金配置器需要的資料

此配置使用 `mvo_ledoit_wolf` 進行資金配置；它是動差估計器，讀取標的價格的滾動視窗，並根據其共變異數產生權重。評估期間開始，並不代表該視窗也重新開始。若只載入留出樣本切片，第一次再平衡便沒有歷史資料可供估計，載入器會退回使用以中位數填補的暖機資料；因此留出樣本最初幾週會按缺乏實際資訊的權重交易，所得夏普比率衡量的將是備援機制。

因此，載入價格時會納入與驗證階段提供給資金配置器相同的回看期間。`setup.yaml` 宣告 `allocator_lookback: 240`，即 240 根八小時價格柱，約八十天。載入器將時間窗起點向前放寬這段期間，但仍將終點限制在標準時間窗的終點；額外的前置資料由滾動視窗使用，不會進入報酬率彙總，因為引擎只在預測組所含的再平衡時間戳記上彙總。

下方的共形預測分支對此配置不起作用；之所以保留，是因為選定配置是解析而來，而非寫死。`conformal_weighted` 依模型已產生的殘差決定部位規模，但在留出樣本上沒有可用的殘差，因為每筆留出樣本報酬率都在受評估的時間窗內實現。因此，共形預測配置會使用驗證殘差校準，並設置涵蓋標籤預測期間的禁運期。此配置則依價格動差分配資金，完全不需要校準。
<!-- ML4T_ZH_END c0006-p000 -->


In [5]:
strategy = strategy_view(json.loads(carrier["spec_json"]))
allocation = strategy.get("allocation") or {}
warmup = strategy_warmup_periods({"allocation": allocation} if allocation else {})
NEEDS_CALIBRATION = allocation.get("method") == "conformal_weighted"
embargo_steps = holdout_conformal_embargo_steps(CASE_STUDY_ID, LABEL) if NEEDS_CALIBRATION else 0
print(f"Allocator {allocation.get('method', 'equal_weight')!r}, warmup {warmup} bars")
if NEEDS_CALIBRATION:
    print(
        f"  conformal configuration: embargo {embargo_steps} observation(s), widths written below."
    )
else:
    print("  needs no calibration; sized from price moments over the warmup window.")

Allocator 'equal_weight', warmup 0 bars
  needs no calibration; sized from price moments over the warmup window.


## 3. The backtest

The strategy specification is the selected configuration's own, re-pointed at the holdout
prediction set and the holdout price window. Nothing else about it changes - the signal, the
allocator and the `time_exit_20` overlay are carried across, and the commission and slippage are
the levels `setup.yaml` declares, the same ones every validation number in this case study was
net of and the same ones sitting inside the swept grid in [`16_costs`](16_costs.ipynb).

The run registers under `stage='holdout'`, which the registry derives from the prediction
set's split rather than from anything asserted here.

The window carries one backtest at a time, for the same reason 17 lets it carry one
prediction generation at a time. 17's guard is on the model - the training identity and the
checkpoint - and it cannot see this one: a changed allocator, overlay or cost level
produces the same holdout predictions and a different result from them.

Two fields are rebuilt rather than carried. `input_identity` records the digests of the data a
run actually read - here the price panel and the official funding settlements - and the selected
configuration's copy describes the validation window. Cloning it produces a record that names
inputs the run never touched, which is exactly what a consumer checking a price digest against
the canonical one would refuse. They are derived from the holdout frames instead.

The funding settlements are also passed to the runner, not merely digested. This case study
is about a cashflow that accrues on holding rather than trading, every validation number in
it is net of the funding the position actually paid or received, and a holdout run that
omitted it would compare a strategy without its central economics against one with it.


<!-- ML4T_ZH_BEGIN c0008-p000 -->
## 3. 回測

策略規格沿用選定配置自身的規格，只將其指向留出樣本預測組及留出樣本價格時間窗。其餘部分完全不變：訊號、資金配置器及 `time_exit_20` 風險覆蓋均原樣沿用；手續費與滑價則採用 `setup.yaml` 宣告的水準，也就是本案例研究每個驗證數值都已扣除的相同成本，以及 [`16_costs`](16_costs.ipynb) 掃描網格中所包含的相同水準。

執行結果登錄於 `stage='holdout'`，此值由登錄庫根據預測組的資料分割推導，而非由此處宣稱。

此時間窗一次只承載一筆回測，理由與第 17 份 Notebook 讓它一次只承載一個預測世代相同。第 17 份 Notebook 的防護檢查針對模型，即訓練識別資訊與檢查點，無法看見此處的變動：若資金配置器、風險覆蓋或成本水準改變，留出樣本預測仍相同，但從中產生的結果會不同。

有兩個欄位需要重建，而非直接沿用。`input_identity` 記錄執行實際讀取資料的摘要值，在此是價格面板及正式資金費結算資料；選定配置中的副本描述的是驗證時間窗。若直接複製，紀錄便會指向此次執行從未讀取的輸入，檢查價格摘要值是否與標準值一致的使用者正會因此拒收。這些摘要值改由留出樣本資料框推導。

資金費結算資料也會傳給執行器，而非僅計算摘要值。本案例研究探討的是持有部位而非交易時累積的現金流；每個驗證數值都已扣除該部位實際支付或收取的資金費。若留出樣本執行時省略資金費，就等於拿缺少核心經濟機制的策略，與包含該機制的策略相比。
<!-- ML4T_ZH_END c0008-p000 -->

The test is the backtest hash, not a field-by-field comparison. Every input that changes
the result is in that hash by construction, and a guard naming fields instead has to be
right about all of them. The hash is resolved before anything runs, so nothing is evaluated
on the holdout before the question is answered, and it comes from `backtest_run_status` -
the call the runner itself makes - so the guard and the runner cannot disagree about
identity. The run asserts they still agree afterwards, because a guard that had quietly
stopped predicting the hash would let everything through while looking correct.

<!-- ML4T_ZH_BEGIN c0008-p001 -->
檢查依據是回測雜湊值，而非逐欄位比較。依設計，每項會改變結果的輸入都包含在該雜湊值內；若防護檢查逐一列出欄位，就必須確保所有欄位都列對。雜湊值在任何執行開始前就解析完成，因此問題回答之前不會在留出樣本上評估任何內容；它來自 `backtest_run_status`，也就是執行器自身呼叫的函式，因此防護檢查與執行器對識別資訊的判定不會相左。執行後仍會斷言兩者一致，因為若防護檢查悄悄失去預測雜湊值的能力，表面上看似正確，實際上卻會放行所有情況。
<!-- ML4T_ZH_END c0008-p001 -->


In [6]:
prices = load_backtest_prices_for(
    CASE_STUDY_ID,
    LABEL,
    split="holdout",
    warmup_periods=warmup,
    max_symbols=MAX_SYMBOLS,
)
predictions = read_predictions(CASE_STUDY_ID, HOLDOUT_PREDICTION_HASH)
funding_rates = funding_rates_for_prices(prices)
print(f"Prices: {len(prices):,} rows, {prices['symbol'].n_unique():,} assets")
print(
    f"Funding: {funding_rates.height:,} settlements over "
    f"{funding_rates['symbol'].n_unique():,} names"
)
print(f"Predictions: {predictions.height:,} rows, {predictions['timestamp'].n_unique():,} stamps")

spec = ensure_backtest_spec(
    CASE_STUDY_ID,
    bt_config,
    json.loads(carrier["spec_json"]),
    prices=prices,
    prediction_hash=HOLDOUT_PREDICTION_HASH,
    initial_cash=bt_config.initial_cash,
)
spec["chapter"] = "ch20"
spec["input_identity"] = {
    **spec.get("input_identity", {}),
    "prices": value_digest(prices),
    "funding_rates": value_digest(funding_rates),
}
if NEEDS_CALIBRATION:
    spec = ensure_conformal_calibration_identity(spec, holdout_embargo_steps=embargo_steps)

spec["backtest_config"]["account"]["allow_short_selling"] = resolved_allow_short_selling(spec, None)
prospective_hash = backtest_run_status(CASE_STUDY_ID, HOLDOUT_PREDICTION_HASH, spec).backtest_hash
superseded_backtests = sorted(
    {
        row["backtest_hash"]
        for row in _registered_holdout_backtests(CASE_DIR, HOLDOUT_PREDICTION_HASH)
    }
    - {prospective_hash}
)
if superseded_backtests and not REPLACE_HOLDOUT:
    raise RuntimeError(
        "the holdout window already carries a backtest of a different configuration: "
        + ", ".join(superseded_backtests)
        + f". This run would register {prospective_hash} and has not run. Set "
        "REPLACE_HOLDOUT=True to discard the earlier one, or leave the selection where it was."
    )
for backtest_hash in superseded_backtests:
    print(f"REPLACING holdout backtest {backtest_hash}")
    _delete_holdout_backtest(CASE_DIR, backtest_hash)

# The guard has passed, so this run will register and any artifact it is sized by belongs
# beside this prediction set. Writing before the guard would replace the calibration a
# registered run was sized by, and the guard could still refuse afterwards.
if NEEDS_CALIBRATION:
    widths = compute_holdout_conformal_widths(
        CASE_STUDY_ID,
        carrier["val_prediction_hash"],
        HOLDOUT_PREDICTION_HASH,
        alpha=float(allocation.get("alpha", 0.2)),
        min_calibration_n=int(allocation["min_calibration_n"]),
        embargo_steps=embargo_steps,
        write=True,
    )
    print(
        f"Conformal widths: {widths.height:,} rows over "
        f"{widths['symbol'].n_unique():,} names, embargo {embargo_steps} observation(s)"
    )

result = run_backtest(
    CASE_STUDY_ID,
    HOLDOUT_PREDICTION_HASH,
    spec,
    prices=prices,
    predictions=predictions,
    label=LABEL,
    funding_rates=funding_rates,
    register=True,
    initial_cash=bt_config.initial_cash,
    calendar=bt_config.calendar,
)
if result.backtest_hash != prospective_hash:
    raise RuntimeError(
        f"the guard predicted {prospective_hash} and the runner registered "
        f"{result.backtest_hash}. The guard decides what may run on the holdout, so a guard "
        "that no longer reproduces the runner's identity is not a smaller problem than the "
        "one it was written for."
    )
print(f"Holdout backtest: {result.backtest_hash}")

Prices: 41,667 rows, 19 assets
Funding: 41,324 settlements over 19 names
Predictions: 40,821 rows, 2,190 stamps
  SKIP backtest (complete (hash=ee7c119e1009)) — reusing cached result
Holdout backtest: ee7c119e1009


## 4. What it came out at

The two numbers below are one strategy measured on two disjoint periods, and the gap
between them is not an estimate of decay. The validation figure is the maximum of a ranking
over thousands of backtests, so it carries the selection; the holdout figure is one
measurement, so it carries its own sampling error. Both push the pair apart before any real
change in the strategy's edge.
[`19_strategy_analysis`](19_strategy_analysis.ipynb) is where they are given intervals and
the selection correction.

<!-- ML4T_ZH_BEGIN c0010-p000 -->
## 4. 結果數值

下方兩個數值是同一策略在兩段互不重疊期間的衡量結果，其差距並非績效衰減的估計值。驗證數值是在數千次回測排名中取得的最大值，因此帶有選擇效應；留出樣本數值則是一次衡量，因此帶有自身的抽樣誤差。即使策略的優勢完全沒有實際變化，兩者也都會拉大這組數值的差距。[`19_strategy_analysis`](19_strategy_analysis.ipynb) 會為兩者提供區間及選擇修正。
<!-- ML4T_ZH_END c0010-p000 -->


In [7]:
metrics = result.metrics
# The selected configuration's own registered Sharpe, not the resolver's. `resolve_solvent_carrier`
# reports the common-support figure, which re-ranks candidates on the timestamps every one of them
# covers; that is the right number for choosing between candidates and the wrong one to set beside a
# holdout measured over its own full window. Both are printed, so neither has to be inferred from
# the other.
with sqlite3.connect(str(CASE_DIR / "run_log" / "registry.db")) as conn:
    carrier_sharpe, carrier_periods = conn.execute(
        "SELECT sharpe, n_periods FROM backtest_metrics WHERE backtest_hash = ?",
        (carrier["val_backtest_hash"],),
    ).fetchone()

print(f"Validation Sharpe over its {int(carrier_periods):,} periods: {carrier_sharpe:.3f}")
print(f"  the same run re-ranked on common support: {carrier['val_sharpe']:.3f}")
print(
    f"Holdout Sharpe over {int(metrics['n_periods']):,} periods:      "
    f"{metrics.get('sharpe', float('nan')):.3f}"
)
print(
    f"Holdout: CAGR {metrics.get('cagr', float('nan')):.1%}, "
    f"max drawdown {metrics.get('max_drawdown', float('nan')):.2%}, "
    f"win rate {metrics.get('win_rate', float('nan')):.0%}"
)

Validation Sharpe over its 729 periods: 1.668
  the same run re-ranked on common support: 1.668
Holdout Sharpe over 731 periods:      -0.832
Holdout: CAGR -44.8%, max drawdown -82.26%, win rate 47%


## What this notebook establishes, and what it does not

It establishes a return series for the selected configuration over a period no choice in
this case study was made on. That is the only thing a holdout can give.

It does not establish that this configuration was the right one to carry here. The
selection that brought it was made on validation, over a pool large enough that its maximum
is optimistic by construction, and this notebook inherits that pool without correcting for
it. The deflation is [`19_strategy_analysis`](19_strategy_analysis.ipynb)'s.

The holdout stays re-runnable. If the selection changes, this generation is deleted and
another is produced; it is not a resource that has been spent.

**Next:** [`19_strategy_analysis`](19_strategy_analysis.ipynb).

<!-- ML4T_ZH_BEGIN c0012-p000 -->
## 本 Notebook 證明了什麼、沒有證明什麼

它建立了選定配置在本案例研究未曾據以做出任何選擇的期間內的報酬率序列。這就是留出樣本所能提供的內容。

它沒有證明將這項配置帶到此處是正確選擇。使它進入此處的選擇是在驗證資料上做出的，而候選池夠大，使其中最大值依設計就偏樂觀。本 Notebook 沿用該候選池，卻未對此修正。縮減偏高估計的工作屬於 [`19_strategy_analysis`](19_strategy_analysis.ipynb)。

留出樣本仍可重新執行。若選擇改變，就刪除這個世代並產生另一個；它不是一次用完的資源。

**下一步：** [`19_strategy_analysis`](19_strategy_analysis.ipynb)。
<!-- ML4T_ZH_END c0012-p000 -->
